# M6-T2 — Train & Serialize the Email Model (LinearSVC + TF-IDF)

**Owner:** Michael Chea  
**Depends on:** M6-T1 — frozen, checksummed train/test splits must be present locally and verified before running this notebook.

### Inputs / Outputs
| | |
|---|---|
| **In** | `data/processed/email_train.csv` / `email_test.csv` (M4-T7 freeze) |
| **Features** | `text_clean` (TF-IDF) + 5 numeric: `urgency_score`, `link_count`, `html_ratio`, `word_count`, `avg_word_length` |
| **Out** | `models/email_linearsvc.joblib` — full pipeline (TF-IDF + scaler + LinearSVC) |
| **S3** | `s3://email-security-pipeline-datasets/models/artifacts/email/email_linearsvc.joblib` |

### Acceptance criteria
- Test F1 ≥ 0.990 (reproduces M5-T2 result)
- TF-IDF vectorizer persisted **inside** the pipeline artifact (inference parity)
- Artifact SHA-256 recorded and uploaded to S3

## Step 0 — Download frozen splits from S3 (skip if already local)
```bash
mkdir -p data/processed
for f in email_train email_test; do
  aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/${f}.csv \
            data/processed/${f}.csv --profile lab-user
done
# verify checksums
aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/SPLITS.sha256 \
          data/processed/SPLITS.sha256 --profile lab-user
cd data/processed && sha256sum -c SPLITS.sha256 --ignore-missing && cd -
```

In [16]:
import hashlib
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'M6' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'notebooks' / 'M5'))

PROC         = REPO_ROOT / 'data' / 'processed'
S3_ARTIFACT  = 's3://email-security-pipeline-datasets/models/artifacts/email/email_linearsvc.joblib'
S3_PROFILE   = 'lab-user'

for f in ('email_train.csv', 'email_test.csv'):
    assert (PROC / f).exists(), f'Missing {f} — run Step 0 first'
print('Split files present.')

Split files present.


## Step 1 — Load frozen splits

In [17]:
from m5_harness import load_email, email_pipeline

etr, ete = load_email()
print(f'Train : {len(etr):>7,} rows  |  label dist: {etr["label"].value_counts().to_dict()}')
print(f'Test  : {len(ete):>7,} rows  |  label dist: {ete["label"].value_counts().to_dict()}')

Train :  65,662 rows  |  label dist: {1: 34276, 0: 31386}
Test  :  16,416 rows  |  label dist: {1: 8569, 0: 7847}


## Step 2 — Train LinearSVC pipeline
Uses the shared M5 harness pipeline: TF-IDF (unigrams + bigrams, max 20k features) + StandardScaler on 5 numeric features + LinearSVC.

In [18]:
from sklearn.svm import LinearSVC

pipe = email_pipeline(LinearSVC(random_state=42))
pipe.fit(etr, etr['label'])
print('Training complete.')

Training complete.


## Step 3 — Serialize artifact
The full pipeline (vectorizer + scaler + classifier) is saved as a single joblib file so inference never needs to reconstruct it.

In [19]:
import joblib

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

models_dir    = REPO_ROOT / 'models'
models_dir.mkdir(exist_ok=True)
artifact_path = models_dir / 'email_linearsvc.joblib'

joblib.dump(pipe, artifact_path)
digest = sha256_file(artifact_path)

print(f'Artifact : {artifact_path}')
print(f'SHA-256  : {digest}')

Artifact : /Users/nara/Documents/EDU/Sem2/CYT300/Project/models/email_linearsvc.joblib
SHA-256  : b7acb294c734cd54d9cd445a4d7d139d65433002b8c7f02fae7cd10ddf857ffc


## Step 4 — Evaluate on held-out test split

In [20]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

preds  = pipe.predict(ete)
scores = pipe.decision_function(ete)   # LinearSVC has no predict_proba

f1 = f1_score(ete['label'], preds)

print('=== Test-set metrics ===')
print(f'  Accuracy  : {accuracy_score(ete["label"], preds):.4f}')
print(f'  Precision : {precision_score(ete["label"], preds):.4f}')
print(f'  Recall    : {recall_score(ete["label"], preds):.4f}')
print(f'  F1        : {f1:.4f}  (target ≥ 0.990)')
print(f'  ROC-AUC   : {roc_auc_score(ete["label"], scores):.4f}')
print()
print(classification_report(ete['label'], preds, target_names=['benign', 'phishing']))

assert f1 >= 0.990, f'F1 {f1:.4f} below target — investigate before uploading artifact'

=== Test-set metrics ===
  Accuracy  : 0.9898
  Precision : 0.9886
  Recall    : 0.9919
  F1        : 0.9903  (target ≥ 0.990)
  ROC-AUC   : 0.9991

              precision    recall  f1-score   support

      benign       0.99      0.99      0.99      7847
    phishing       0.99      0.99      0.99      8569

    accuracy                           0.99     16416
   macro avg       0.99      0.99      0.99     16416
weighted avg       0.99      0.99      0.99     16416



## Step 5 — Upload artifact to S3
Uploads to the `models/artifacts/email/` path so M6-T6 (inference wrapper) and M6-T7 (artifact registry) can find it.

In [21]:
print(f'Uploading to {S3_ARTIFACT} ...')
result = subprocess.run(
    ['aws', 's3', 'cp', str(artifact_path), S3_ARTIFACT, '--profile', S3_PROFILE],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print('Upload complete ✅')
    print(f'Verify: aws s3 ls {S3_ARTIFACT} --profile {S3_PROFILE}')
else:
    print(f'Upload failed ⚠️\n{result.stderr}')
    print('Run manually:')
    print(f'  aws s3 cp {artifact_path} {S3_ARTIFACT} --profile {S3_PROFILE}')

Uploading to s3://email-security-pipeline-datasets/models/artifacts/email/email_linearsvc.joblib ...
Upload complete ✅
Verify: aws s3 ls s3://email-security-pipeline-datasets/models/artifacts/email/email_linearsvc.joblib --profile lab-user


## Summary

In [22]:
print('=' * 55)
print('M6-T2 EMAIL MODEL SUMMARY')
print('=' * 55)
print(f'Model    : LinearSVC (random_state=42)')
print(f'Pipeline : TF-IDF (20k, 1-2gram) + StandardScaler + LinearSVC')
print(f'Features : text_clean + urgency_score, link_count, html_ratio, word_count, avg_word_length')
print(f'Train    : {len(etr):,} rows')
print(f'Test F1  : {f1:.4f}  {"✅" if f1 >= 0.990 else "❌ below target"}')
print(f'Artifact : {artifact_path.name}')
print(f'SHA-256  : {digest}')
print(f'S3       : {S3_ARTIFACT}')
print('=' * 55)

M6-T2 EMAIL MODEL SUMMARY
Model    : LinearSVC (random_state=42)
Pipeline : TF-IDF (20k, 1-2gram) + StandardScaler + LinearSVC
Features : text_clean + urgency_score, link_count, html_ratio, word_count, avg_word_length
Train    : 65,662 rows
Test F1  : 0.9903  ✅
Artifact : email_linearsvc.joblib
SHA-256  : b7acb294c734cd54d9cd445a4d7d139d65433002b8c7f02fae7cd10ddf857ffc
S3       : s3://email-security-pipeline-datasets/models/artifacts/email/email_linearsvc.joblib
